## FIFA World Cup 2026 — ML Prediction Pipeline (v2)
**Author:** Iyinoluwa Don-Taiwo  
**Model:** Logistic Regression (hyperparameter-tuned)  
**Key upgrades over v1:** Weighted Elo · Form-5 & Form-10 · Goals scored/conceded · Tournament importance · Calibration · Confusion matrix · Proper WC tiebreakers · Host-nation advantage

---
### Pipeline Overview
1. Imports  
2. Load Data  
3. Exploratory Data Analysis (EDA)  
4. Feature Engineering (10 → 20 features)  
5. Model Building — Tuning + Calibration  
6. Evaluation — Log-loss · Accuracy · Confusion Matrix · Classification Report  
7. Tournament Simulation — Monte Carlo (10,000 runs)  
8. Visualizations  


## 1. Imports

In [ ]:
import os
import random
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import defaultdict

# Preprocessing
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score,
    log_loss,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

warnings.filterwarnings('ignore')
np.random.seed(42)
random.seed(42)
print("All packages imported successfully.")


## 2. Load Data

In [ ]:
df = pd.read_csv("data/master_matches.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print(f"Shape: {df.shape}")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"\nColumns:\n{list(df.columns)}")


In [ ]:
df.head()


In [ ]:
print("Missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f"\nResult distribution:")
print(df["result"].value_counts())


## 3. Exploratory Data Analysis (EDA)

In [ ]:
print("=== Basic Statistics ===")
print(df[["home_score", "away_score", "home_ranking_pts", "away_ranking_pts"]].describe().round(2))

print("\n=== Top Tournaments ===")
print(df["tournament"].value_counts().head(10))

print("\n=== Confederations ===")
print(df["home_confederation"].value_counts())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("EDA — Match Distributions", fontsize=14, fontweight="bold")

# Result distribution
result_labels = {1: "Home Win", 0: "Draw", -1: "Away Win"}
counts = df["result"].map(result_labels).value_counts()
colors = ["#2ecc71", "#95a5a6", "#e74c3c"]
axes[0].bar(counts.index, counts.values, color=colors, edgecolor="white")
axes[0].set_title("Match Outcome Distribution")
axes[0].set_ylabel("Count")
for bar, val in zip(axes[0].patches, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 f"{val:,}\n({val/len(df)*100:.1f}%)", ha="center", fontsize=9)

# Goals distribution
axes[1].hist(df["home_score"], bins=range(0, 10), alpha=0.7, label="Home", color="#3498db", edgecolor="white")
axes[1].hist(df["away_score"], bins=range(0, 10), alpha=0.7, label="Away", color="#e67e22", edgecolor="white")
axes[1].set_title("Goals Scored Distribution")
axes[1].set_xlabel("Goals"); axes[1].set_ylabel("Frequency")
axes[1].legend()

# Ranking pts distribution
axes[2].hist(df["home_ranking_pts"].dropna(), bins=40, color="#9b59b6", edgecolor="white", alpha=0.8)
axes[2].set_title("FIFA Ranking Points Distribution")
axes[2].set_xlabel("Points"); axes[2].set_ylabel("Frequency")

plt.tight_layout()
plt.savefig("outputs/eda_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved → outputs/eda_distributions.png")


In [ ]:
# Results by confederation
conf_results = df.groupby("home_confederation")["result"].value_counts(normalize=True).unstack().fillna(0)
conf_results = conf_results.rename(columns={1:"Home Win", 0:"Draw", -1:"Away Win"})
conf_results[["Home Win","Draw","Away Win"]].plot(
    kind="bar", stacked=True, figsize=(10, 4),
    color=["#2ecc71","#95a5a6","#e74c3c"], edgecolor="white"
)
plt.title("Win/Draw/Loss Rate by Confederation (Home Team)")
plt.ylabel("Proportion"); plt.xticks(rotation=45, ha="right")
plt.legend(loc="upper right"); plt.tight_layout()
plt.savefig("outputs/eda_confederation.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Feature Engineering

**10 new features added in v2 (20 total):**

| Feature | v1 | v2 | What it captures |
|---|---|---|---|
| `elo_diff` | Fixed K=40 | Weighted K (25–60) | Dynamic team strength, importance-aware |
| `ranking_diff` | ✅ | ✅ | FIFA official team strength gap |
| `home_form_5` | ✅ | ✅ | Short-term momentum |
| `away_form_5` | ✅ | ✅ | Short-term momentum |
| `form_diff` | ✅ | ✅ | Relative form gap |
| `home_form_10` | ❌ | ✅ | Medium-term trend |
| `away_form_10` | ❌ | ✅ | Medium-term trend |
| `form_diff_10` | ❌ | ✅ | Relative medium-term gap |
| `home_goals_scored_5` | ❌ | ✅ | Attacking quality |
| `home_goals_conceded_5` | ❌ | ✅ | Defensive solidity |
| `away_goals_scored_5` | ❌ | ✅ | Attacking quality |
| `away_goals_conceded_5` | ❌ | ✅ | Defensive solidity |
| `goal_diff_scored` | ❌ | ✅ | Combined attacking signal |
| `goal_diff_conceded` | ❌ | ✅ | Combined defensive signal |
| `h2h_home_wins` | ✅ | ✅ | Historical head-to-head |
| `h2h_draws` | ✅ | ✅ | Historical head-to-head |
| `h2h_home_losses` | ✅ | ✅ | Historical head-to-head |
| `is_neutral` | ✅ | ✅ | Venue context |
| `same_confederation` | ✅ | ✅ | Familiarity context |
| `tournament_importance` | ❌ | ✅ | Match stakes (WC=3, major=2, qual=1) |


### 4a. Weighted Elo Ratings

In [ ]:
# Weighted Elo — K varies by tournament importance.
# World Cup K=60: results move ratings more than qualifiers (K=25).
# This makes the ratings more sensitive to important matches.

TOURNAMENT_WEIGHTS = {
    'FIFA World Cup': 60,
    'UEFA Euro': 50,
    'Copa America': 50,
    'African Cup of Nations': 50,
    'AFC Asian Cup': 50,
    'Gold Cup': 40,
    'CONCACAF Nations League': 35,
    'UEFA Nations League': 35,
    'UEFA Euro qualification': 30,
    'FIFA World Cup qualification': 30,
    'African Cup of Nations qualification': 25,
    'AFC Asian Cup qualification': 25,
}

def get_k(tournament):
    """Return the K-factor for a given tournament. Higher = more important."""    for key, k in TOURNAMENT_WEIGHTS.items():
        if key.lower() in tournament.lower():
            return k
    return 25  # default for minor competitions

elo = defaultdict(lambda: 1500.0)
home_elo_w, away_elo_w = [], []

for _, row in df.iterrows():
    home, away = row["home_team"], row["away_team"]
    he, ae = elo[home], elo[away]

    # Record Elo BEFORE this match — no data leakage
    home_elo_w.append(he)
    away_elo_w.append(ae)

    # Expected score for home team
    exp_h = 1 / (1 + 10 ** ((ae - he) / 400))
    sh = 1.0 if row["result"] == 1 else (0.5 if row["result"] == 0 else 0.0)

    # Update with tournament-weighted K
    k = get_k(row["tournament"])
    elo[home] = he + k * (sh - exp_h)
    elo[away] = ae + k * ((1 - sh) - (1 - exp_h))

df["home_elo_w"] = home_elo_w
df["away_elo_w"] = away_elo_w
df["elo_diff_w"] = df["home_elo_w"] - df["away_elo_w"]

# Save the final Elo state (needed for tournament simulation)
elo_state_final = dict(elo)

print("Weighted Elo computed.")
print(f"Elo range: {df['home_elo_w'].min():.0f} – {df['home_elo_w'].max():.0f}")
print(f"\nTop 10 teams by final Elo:")
top_elo = sorted(elo_state_final.items(), key=lambda x: -x[1])[:10]
for rank, (team, rating) in enumerate(top_elo, 1):
    print(f"  {rank:>2}. {team:<25} {rating:.0f}")


### 4b. Tournament Importance Encoding

In [ ]:
def get_tournament_importance(tournament):
    """
    Encode how important a match is by competition type.
    3 = World Cup knockout
    2 = Major tournament (Euros, Copa, AFCON, Asian Cup, Gold Cup)
    1 = Nations League or Qualification
    0 = Other
    """
    t = tournament.lower()
    if "world cup" in t and "qualif" not in t:
        return 3
    if any(x in t for x in ["euro", "copa", "cup of nations", "asian cup", "gold cup"])        and "qualif" not in t:
        return 2
    if "nations league" in t or "qualif" in t:
        return 1
    return 0

df["tournament_importance"] = df["tournament"].apply(get_tournament_importance)

print("Tournament importance distribution:")
print(df["tournament_importance"].value_counts().sort_index())


### 4c. Extended Form Windows (5 and 10 matches) + Goals

In [ ]:
# For each match, compute:
# - Win rate in last 5 and 10 matches (for both teams)
# - Average goals scored and conceded in last 5 matches (for both teams)
#
# CRITICAL: All computations use only matches BEFORE the current date.
# Using the current match or future matches would be data leakage.

home_form_10, away_form_10 = [], []
home_gs5, home_gc5 = [], []
away_gs5, away_gc5 = [], []

for i, (idx, row) in enumerate(df.iterrows()):
    if i % 3000 == 0:
        print(f"  Processing row {i}/{len(df)}...")

    date, ht, at = row["date"], row["home_team"], row["away_team"]

    # All past matches for each team (before this match)
    hh = df[(df["home_team"] == ht) & (df["date"] < date)]
    ha = df[(df["away_team"] == ht) & (df["date"] < date)]
    ah = df[(df["home_team"] == at) & (df["date"] < date)]
    aa = df[(df["away_team"] == at) & (df["date"] < date)]

    # Form-10: combine home and away matches, sort by date, take last 10
    def form10(home_matches, away_matches):
        hw = home_matches[["date"]].assign(
            w=home_matches["result"].map(lambda x: 1 if x == 1 else 0))
        aw = away_matches[["date"]].assign(
            w=away_matches["result"].map(lambda x: 1 if x == -1 else 0))
        combined = pd.concat([hw, aw]).sort_values("date").tail(10)
        return combined["w"].mean() if len(combined) else 0.5

    home_form_10.append(form10(hh, ha))
    away_form_10.append(form10(ah, aa))

    # Goals: last 5 matches for each team
    hh5 = hh.tail(5); ha5 = ha.tail(5)
    ah5 = ah.tail(5); aa5 = aa.tail(5)

    gs_h = list(hh5["home_score"]) + list(ha5["away_score"])
    gc_h = list(hh5["away_score"]) + list(ha5["home_score"])
    gs_a = list(ah5["home_score"]) + list(aa5["away_score"])
    gc_a = list(ah5["away_score"]) + list(aa5["home_score"])

    home_gs5.append(np.mean(gs_h) if gs_h else 1.2)
    home_gc5.append(np.mean(gc_h) if gc_h else 1.0)
    away_gs5.append(np.mean(gs_a) if gs_a else 1.2)
    away_gc5.append(np.mean(gc_a) if gc_a else 1.0)

df["home_form_10"] = home_form_10
df["away_form_10"] = away_form_10
df["form_diff_10"] = df["home_form_10"] - df["away_form_10"]

df["home_goals_scored_5"]   = home_gs5
df["home_goals_conceded_5"] = home_gc5
df["away_goals_scored_5"]   = away_gs5
df["away_goals_conceded_5"] = away_gc5
df["goal_diff_scored"]   = df["home_goals_scored_5"] - df["away_goals_scored_5"]
df["goal_diff_conceded"] = df["away_goals_conceded_5"] - df["home_goals_conceded_5"]

print("\nFeature engineering complete.")
print(df[["home_form_10", "away_form_10", "home_goals_scored_5",
          "home_goals_conceded_5", "elo_diff_w", "tournament_importance"]].describe().round(3))


### 4d. Head-to-Head and Context Features

In [ ]:
# These features were already in v1 — carried forward unchanged.
# h2h computed by looking at all past meetings between two teams before the current match.

h2h_wins, h2h_draws, h2h_losses = [], [], []

for _, row in df.iterrows():
    past = df[
        (df["home_team"] == row["home_team"]) &
        (df["away_team"] == row["away_team"]) &
        (df["date"] < row["date"])
    ]
    h2h_wins.append(int((past["result"] == 1).sum()))
    h2h_draws.append(int((past["result"] == 0).sum()))
    h2h_losses.append(int((past["result"] == -1).sum()))

df["h2h_home_wins"]   = h2h_wins
df["h2h_draws"]       = h2h_draws
df["h2h_home_losses"] = h2h_losses
df["ranking_diff"]    = df["home_ranking_pts"] - df["away_ranking_pts"]
df["is_neutral"]      = df["neutral"].astype(int)
df["same_confederation"] = (
    df["home_confederation"] == df["away_confederation"]
).astype(int)

print("Context features computed.")
print(df[["h2h_home_wins","h2h_draws","h2h_home_losses",
          "ranking_diff","is_neutral","same_confederation"]].describe().round(2))


### 4e. Final Feature Set

In [ ]:
FEATURES = [
    "elo_diff_w",              # Weighted Elo gap (NEW — replaces elo_diff)
    "ranking_diff",            # FIFA points gap
    "home_form_5",             # Short-term home form
    "away_form_5",             # Short-term away form
    "form_diff",               # Short-term form gap
    "home_form_10",            # Medium-term home form (NEW)
    "away_form_10",            # Medium-term away form (NEW)
    "form_diff_10",            # Medium-term form gap (NEW)
    "home_goals_scored_5",     # Home attack quality (NEW)
    "home_goals_conceded_5",   # Home defensive solidity (NEW)
    "away_goals_scored_5",     # Away attack quality (NEW)
    "away_goals_conceded_5",   # Away defensive solidity (NEW)
    "goal_diff_scored",        # Relative attacking gap (NEW)
    "goal_diff_conceded",      # Relative defensive gap (NEW)
    "h2h_home_wins",           # Head-to-head history
    "h2h_draws",
    "h2h_home_losses",
    "is_neutral",              # Venue context
    "same_confederation",      # Confederation context
    "tournament_importance",   # Match stakes (NEW)
]

print(f"Total features: {len(FEATURES)}")
print(f"\nFeature summary:")
print(df[FEATURES].describe().round(3))


## 5. Model Building

**Three-phase approach:**
1. Encode target variable
2. Time-based train / calibration / test split
3. Train 4 models → calibrate → evaluate → ensemble


In [ ]:
# Encode target: result (-1/0/1) → integer labels for sklearn
le = LabelEncoder()
df["result_encoded"] = le.fit_transform(df["result"])

# le.classes_ = [-1, 0, 1] → encoded as [0, 1, 2]
print("Label encoder classes:", le.classes_)
print("Encoded mapping:", dict(zip(le.classes_, le.transform(le.classes_))))


In [ ]:
# Time-based split — NEVER split randomly on time-series data.
# Random splits cause data leakage: future matches influence past model training.
#
# Train  : before 2018  (historical — model learns from this)
# Cal    : 2018–2021    (calibration holdout — corrects probability outputs)
# Test   : 2022 onward  (genuinely unseen — honest evaluation)

train_df = df[df["date"] < "2018-01-01"]
cal_df   = df[(df["date"] >= "2018-01-01") & (df["date"] < "2022-01-01")]
test_df  = df[df["date"] >= "2022-01-01"]

X_train = train_df[FEATURES].fillna(0)
y_train = train_df["result_encoded"]
X_cal   = cal_df[FEATURES].fillna(0)
y_cal   = cal_df["result_encoded"]
X_test  = test_df[FEATURES].fillna(0)
y_test  = test_df["result_encoded"]

print(f"Train : {len(X_train):,} matches (before 2018)")
print(f"Cal   : {len(X_cal):,} matches (2018–2021)")
print(f"Test  : {len(X_test):,} matches (2022 onward)")


### 5a. Hyperparameter Tuning — Logistic Regression

In [ ]:
# TimeSeriesSplit preserves temporal order in cross-validation.
# Standard KFold would leak future data into past folds.
# GridSearchCV finds the best regularisation strength C.

tscv = TimeSeriesSplit(n_splits=5)

lr_grid = GridSearchCV(
    LogisticRegression(solver="lbfgs", max_iter=2000),
    param_grid={"C": [0.01, 0.05, 0.1, 0.5, 1, 5, 10]},
    cv=tscv,
    scoring="neg_log_loss",
    n_jobs=-1,
)
lr_grid.fit(X_train, y_train)
best_lr = lr_grid.best_estimator_

print(f"Best C: {lr_grid.best_params_['C']}")
print(f"Best CV log-loss: {-lr_grid.best_score_:.4f}")


### 5b. Train All Models

In [ ]:
print("Training XGBoost...")
xgb_m = XGBClassifier(
    objective="multi:softprob", num_class=3, eval_metric="mlogloss",
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0,
)
xgb_m.fit(X_train, y_train)

print("Training LightGBM...")
lgb_m = LGBMClassifier(
    objective="multiclass", num_class=3, metric="multi_logloss",
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1,
)
lgb_m.fit(X_train, y_train)

print("Training Random Forest...")
rf_m = RandomForestClassifier(
    n_estimators=200, max_depth=8, random_state=42, n_jobs=-1,
)
rf_m.fit(X_train, y_train)

print("All models trained.")


### 5c. Calibration

In [ ]:
# Calibration corrects systematic biases in predicted probabilities.
# A model that says 80% win should be right 80% of the time.
# We fit the calibrator on the 2018-2021 holdout set (not the training set).
# method='isotonic' is non-parametric — better than Platt scaling for this data size.

print("Calibrating models on 2018–2021 holdout set...")
lr_c  = CalibratedClassifierCV(best_lr, method="isotonic", cv=None, ensemble=False)
xgb_c = CalibratedClassifierCV(xgb_m,  method="isotonic", cv=None, ensemble=False)
lgb_c = CalibratedClassifierCV(lgb_m,  method="isotonic", cv=None, ensemble=False)
rf_c  = CalibratedClassifierCV(rf_m,   method="isotonic", cv=None, ensemble=False)

lr_c.fit(X_cal, y_cal)
xgb_c.fit(X_cal, y_cal)
lgb_c.fit(X_cal, y_cal)
rf_c.fit(X_cal, y_cal)

print("Calibration complete.")


## 6. Evaluation

**Metrics used:**
- **Log-loss** (primary) — measures probability calibration quality. Lower is better. This is the metric that matters for the simulation system.
- **Accuracy** — overall label correctness. Included for interpretability but not the primary metric.
- **Confusion matrix** — shows exactly which classes the model confuses with which. Critical for understanding where predictions fail.
- **Classification report** — precision, recall, F1 per class. Exposes the draw-prediction weakness explicitly.


In [ ]:
print(f"{'Model':<35} {'Accuracy':>10} {'Log-Loss':>10}")
print("-" * 57)

model_store = {}
for name, m in [
    ("LR (original)",              best_lr),
    ("LR (calibrated)",            lr_c),
    ("XGBoost (calibrated)",       xgb_c),
    ("LightGBM (calibrated)",      lgb_c),
    ("Random Forest (calibrated)", rf_c),
]:
    probs = m.predict_proba(X_test)
    preds = m.predict(X_test)
    ll    = log_loss(y_test, probs)
    acc   = accuracy_score(y_test, preds)
    print(f"  {name:<33} {acc:>10.4f} {ll:>10.4f}")
    model_store[name] = {"model": m, "ll": ll, "acc": acc, "preds": preds, "probs": probs}

# Soft-vote ensemble
ens_probs = np.mean([
    lr_c.predict_proba(X_test),
    xgb_c.predict_proba(X_test),
    lgb_c.predict_proba(X_test),
    rf_c.predict_proba(X_test),
], axis=0)
ens_preds = np.argmax(ens_probs, axis=1)
ens_ll  = log_loss(y_test, ens_probs)
ens_acc = accuracy_score(y_test, ens_preds)
print(f"  {'Soft-Vote Ensemble':<33} {ens_acc:>10.4f} {ens_ll:>10.4f}")

print(f"\n  v1 baseline (LR, no tuning): accuracy=0.5982 | log-loss=0.8823")

best_name  = min(model_store, key=lambda k: model_store[k]["ll"])
best_model = model_store[best_name]["model"]
print(f"\n  Best model: {best_name}")


### 6a. Confusion Matrix

In [ ]:
# The confusion matrix shows WHERE the model makes mistakes.
# Rows = actual class, Columns = predicted class.
# Diagonal = correct predictions.
# Off-diagonal = misclassifications.
#
# For football prediction, draws are almost always misclassified —
# either called as home wins or away wins. The matrix shows this explicitly.

class_names = ["Away Win", "Draw", "Home Win"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Confusion Matrix — Best Model vs Ensemble", fontsize=13, fontweight="bold")

for ax, (title, preds) in zip(axes, [
    (f"Best: {best_name}", model_store[best_name]["preds"]),
    ("Soft-Vote Ensemble", ens_preds),
]):
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(title, fontsize=11)

    # Annotate with percentages
    total = cm.sum()
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            pct = cm[i, j] / cm[i].sum() * 100
            ax.text(j, i + 0.35, f"({pct:.0f}%)", ha="center", va="center",
                    fontsize=8, color="white" if cm[i,j] > cm.max()/2 else "gray")

plt.tight_layout()
plt.savefig("outputs/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nKey observation: Draw row is almost entirely off-diagonal.")
print("The model rarely predicts draws correctly — this is a known ceiling in football ML.")
print("Draws depend on in-match randomness that no pre-match model can see.")


### 6b. Classification Report

In [ ]:
print("=== Classification Report — Best Model ===")
print(classification_report(
    y_test,
    model_store[best_name]["preds"],
    target_names=class_names,
))

print("\n=== Classification Report — Ensemble ===")
print(classification_report(y_test, ens_preds, target_names=class_names))

print("""
Key takeaways:
- Home Win and Away Win: reasonable F1 scores (~0.65–0.70)
- Draw: F1 ~0.07 — the model almost never correctly identifies draws
- This is not a failure of the model — it reflects the sport's inherent unpredictability
- Draws are the minority class AND the most random outcome in football
""")


### 6c. Feature Importance (LR Coefficients)

In [ ]:
# For Logistic Regression, feature importance is read from the coefficients.
# Larger absolute coefficient = that feature pushes the prediction harder.
# We show coefficients for the Home Win class (class index 2).

coefficients = best_lr.coef_[2]  # Home Win class
feat_imp = pd.DataFrame({
    "Feature": FEATURES,
    "Coefficient": coefficients,
}).sort_values("Coefficient", key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(9, 7))
colors = ["#2ecc71" if c > 0 else "#e74c3c" for c in feat_imp["Coefficient"]]
ax.barh(feat_imp["Feature"], feat_imp["Coefficient"], color=colors, edgecolor="white", height=0.6)
ax.axvline(0, color="white", linewidth=0.8, alpha=0.5)
ax.set_xlabel("Coefficient (Home Win class)")
ax.set_title("Feature Importance — Logistic Regression Coefficients\n"
             "Green = pushes toward Home Win | Red = pushes away from Home Win",
             fontsize=11)
ax.tick_params(axis="y", labelsize=9)
plt.tight_layout()
plt.savefig("outputs/feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nTop 5 most influential features:")
print(feat_imp.head(5).to_string(index=False))


### 6d. Save Model

In [ ]:
# Save everything the Streamlit app needs:
# - best single model (for fallback)
# - all calibrated models (for ensemble)
# - label encoder
# - feature list
# - final Elo state (for simulation)
# - full engineered dataframe

use_ensemble = (ens_ll < model_store[best_name]["ll"])
print(f"Use ensemble: {use_ensemble}")
print(f"Best single model: {best_name}")

with open("models/best_model_v2.pkl", "wb") as f:
    pickle.dump({
        "model":            best_model,
        "models_ensemble":  [lr_c, xgb_c, lgb_c, rf_c],
        "use_ensemble":     use_ensemble,
        "le":               le,
        "features":         FEATURES,
        "elo_state":        elo_state_final,
        "df":               df,
        "model_results": {
            "LR_original":   {"acc": model_store["LR (original)"]["acc"],   "ll": model_store["LR (original)"]["ll"]},
            "LR_calibrated": {"acc": model_store["LR (calibrated)"]["acc"], "ll": model_store["LR (calibrated)"]["ll"]},
            "XGB":           {"acc": model_store["XGBoost (calibrated)"]["acc"],       "ll": model_store["XGBoost (calibrated)"]["ll"]},
            "LGB":           {"acc": model_store["LightGBM (calibrated)"]["acc"],      "ll": model_store["LightGBM (calibrated)"]["ll"]},
            "RF":            {"acc": model_store["Random Forest (calibrated)"]["acc"], "ll": model_store["Random Forest (calibrated)"]["ll"]},
            "Ensemble":      {"acc": ens_acc, "ll": ens_ll},
        },
    }, f)

print("\nModel saved → models/best_model_v2.pkl")


## 7. Tournament Simulation — Monte Carlo

**Why Monte Carlo and not a single bracket prediction?**

Picking one outcome per match is overconfident. A team with 70% win probability loses 30% of the time. Running 10,000 full tournament simulations samples outcomes probabilistically — upsets happen naturally. The win% for each team reflects genuine uncertainty across all possible tournament paths.

**Improvements over v1:**
- Proper FIFA tiebreaker rules (H2H points → H2H GD → overall GD → GF → random)
- Host-nation home advantage for USA, Canada, Mexico (K=60 Elo on WC results)
- Goals-based score simulation in group stage for realistic GD calculation


In [ ]:
WC2026_GROUPS = {
    "A": ["Mexico",      "South Africa", "Korea Republic", "New Zealand"],
    "B": ["Canada",      "Switzerland",  "Qatar",          "Honduras"],
    "C": ["Brazil",      "Morocco",      "Haiti",          "Scotland"],
    "D": ["USA",         "Paraguay",     "Australia",      "Cameroon"],
    "E": ["Germany",     "Curacao",      "Ivory Coast",    "Ecuador"],
    "F": ["Netherlands", "Japan",        "Tunisia",        "Senegal"],
    "G": ["Belgium",     "Egypt",        "Iran",           "Costa Rica"],
    "H": ["Spain",       "Cape Verde",   "Saudi Arabia",   "Uruguay"],
    "I": ["France",      "Algeria",      "Norway",         "Costa Rica"],
    "J": ["Argentina",   "Algeria",      "Austria",        "Jordan"],
    "K": ["Portugal",    "Colombia",     "Uzbekistan",     "Thailand"],
    "L": ["England",     "Croatia",      "Ghana",          "Panama"],
}
ALL_TEAMS = list({t for grp in WC2026_GROUPS.values() for t in grp})
HOST_NATIONS = {"USA", "Canada", "Mexico"}
print(f"Total teams: {len(ALL_TEAMS)}")


In [ ]:
CONFEDERATION_MAP = {
    "Spain":"UEFA","France":"UEFA","England":"UEFA","Germany":"UEFA",
    "Netherlands":"UEFA","Belgium":"UEFA","Portugal":"UEFA","Croatia":"UEFA",
    "Switzerland":"UEFA","Norway":"UEFA","Scotland":"UEFA","Austria":"UEFA",
    "Brazil":"CONMEBOL","Argentina":"CONMEBOL","Colombia":"CONMEBOL",
    "Uruguay":"CONMEBOL","Ecuador":"CONMEBOL","Paraguay":"CONMEBOL",
    "Morocco":"CAF","Senegal":"CAF","Egypt":"CAF","Ivory Coast":"CAF",
    "Ghana":"CAF","Cape Verde":"CAF","South Africa":"CAF","Tunisia":"CAF","Algeria":"CAF","Cameroon":"CAF",
    "Japan":"AFC","Korea Republic":"AFC","Iran":"AFC","Saudi Arabia":"AFC",
    "Australia":"AFC","Uzbekistan":"AFC","Jordan":"AFC","Qatar":"AFC","Thailand":"AFC",
    "Mexico":"CONCACAF","USA":"CONCACAF","Canada":"CONCACAF",
    "Panama":"CONCACAF","Curacao":"CONCACAF","Haiti":"CONCACAF","Honduras":"CONCACAF",
    "Costa Rica":"CONCACAF","New Zealand":"OFC",
}

def build_team_stats(df, elo_state):
    """Build a lookup of each team's current stats for the simulation."""    stats = {}
    for team in ALL_TEAMS:
        hm = df[df["home_team"] == team]
        am = df[df["away_team"] == team]
        hm5 = hm.tail(5); am5 = am.tail(5)

        hw5 = list(hm5["result"].map(lambda x: 1 if x==1 else 0))
        aw5 = list(am5["result"].map(lambda x: 1 if x==-1 else 0))
        all5 = sorted(zip(list(hm5["date"])+list(am5["date"]), hw5+aw5))[-5:]
        form5 = np.mean([w for _,w in all5]) if all5 else 0.45

        hw10 = hm[["date"]].assign(w=hm["result"].map(lambda x:1 if x==1 else 0))
        aw10 = am[["date"]].assign(w=am["result"].map(lambda x:1 if x==-1 else 0))
        all10 = pd.concat([hw10,aw10]).sort_values("date").tail(10)
        form10 = all10["w"].mean() if len(all10) else 0.45

        gs = list(hm5["home_score"]) + list(am5["away_score"])
        gc = list(hm5["away_score"]) + list(am5["home_score"])

        pts = 1200.0
        if len(hm): pts = float(hm.iloc[-1]["home_ranking_pts"])
        elif len(am): pts = float(am.iloc[-1]["away_ranking_pts"])

        stats[team] = {
            "elo": elo_state.get(team, 1500.0),
            "pts": pts,
            "form5": form5,
            "form10": form10,
            "goals_scored5":   np.mean(gs) if gs else 1.2,
            "goals_conceded5": np.mean(gc) if gc else 1.0,
            "confederation": CONFEDERATION_MAP.get(team, "UEFA"),
        }
    return stats

team_stats = build_team_stats(df, elo_state_final)
print("Team stats built for", len(team_stats), "teams")
print("\nSample — France:", team_stats.get("France", {}))


In [ ]:
def make_feature_row(home, away, neutral=True):
    """Build a single-row DataFrame with all 20 features for a matchup."""    hs  = team_stats.get(home, {"elo":1500,"pts":1200,"form5":0.45,"form10":0.45,
                                "goals_scored5":1.2,"goals_conceded5":1.0,"confederation":"UEFA"})
    as_ = team_stats.get(away, {"elo":1500,"pts":1200,"form5":0.45,"form10":0.45,
                                 "goals_scored5":1.2,"goals_conceded5":1.0,"confederation":"UEFA"})
    past = df[(df["home_team"]==home)&(df["away_team"]==away)]
    same_c = 1 if hs["confederation"] == as_["confederation"] else 0
    # Host nations are not on neutral ground
    is_n = 0 if (home in HOST_NATIONS and not neutral) else int(neutral)

    return {
        "elo_diff_w":            hs["elo"] - as_["elo"],
        "ranking_diff":          hs["pts"] - as_["pts"],
        "home_form_5":           hs["form5"],
        "away_form_5":           as_["form5"],
        "form_diff":             hs["form5"] - as_["form5"],
        "home_form_10":          hs["form10"],
        "away_form_10":          as_["form10"],
        "form_diff_10":          hs["form10"] - as_["form10"],
        "home_goals_scored_5":   hs["goals_scored5"],
        "home_goals_conceded_5": hs["goals_conceded5"],
        "away_goals_scored_5":   as_["goals_scored5"],
        "away_goals_conceded_5": as_["goals_conceded5"],
        "goal_diff_scored":      hs["goals_scored5"] - as_["goals_scored5"],
        "goal_diff_conceded":    as_["goals_conceded5"] - hs["goals_conceded5"],
        "h2h_home_wins":         int((past["result"]==1).sum()),
        "h2h_draws":             int((past["result"]==0).sum()),
        "h2h_home_losses":       int((past["result"]==-1).sum()),
        "is_neutral":            is_n,
        "same_confederation":    same_c,
        "tournament_importance": 3,
    }

print("Feature builder ready.")


In [ ]:
# Batch precompute all matchup probabilities BEFORE the simulation loop.
# Running model.predict_proba() inside 10,000 loops would be extremely slow.
# Precomputing once and using a dictionary lookup reduces runtime from minutes to seconds.

print("Precomputing all matchup probabilities...")
rows, pairs = [], []
for h in ALL_TEAMS:
    for a in ALL_TEAMS:
        if h != a:
            rows.append(make_feature_row(h, a, neutral=True))
            pairs.append((h, a))

batch = best_model.predict_proba(pd.DataFrame(rows)[FEATURES]).astype(float)
batch /= batch.sum(axis=1, keepdims=True)

# le.classes_ = [-1, 0, 1] → indices [0, 1, 2]
idx_aw = list(le.classes_).index(-1)
idx_d  = list(le.classes_).index(0)
idx_hw = list(le.classes_).index(1)

MATCHUPS = {
    (h, a): (batch[i][idx_hw], batch[i][idx_d], batch[i][idx_aw])
    for i, (h, a) in enumerate(pairs)
}
print(f"Precomputed {len(MATCHUPS):,} matchup pairs.")


In [ ]:
def simulate_group(teams):
    """
    Simulate one group stage round-robin with proper FIFA tiebreakers.
    Tiebreaker order: overall pts → H2H pts → H2H GD → overall GD → GF → random
    """
    pts = {t:0 for t in teams}; gd = {t:0 for t in teams}
    gf  = {t:0 for t in teams}; ga = {t:0 for t in teams}
    h2h_pts = defaultdict(int); h2h_gd = defaultdict(int)

    for i in range(len(teams)):
        for j in range(i+1, len(teams)):
            h, a = teams[i], teams[j]
            hw, d, aw = MATCHUPS.get((h,a), (0.4,0.2,0.4))
            r = random.random()

            if r < hw:    # Home win — simulate realistic scoreline
                hs_ = random.randint(1, 3)
                as_ = random.randint(0, max(0, hs_-1))
                pts[h] += 3; h2h_pts[h] += 3
            elif r < hw+d:  # Draw
                hs_ = random.randint(0, 2); as_ = hs_
                pts[h] += 1; pts[a] += 1
                h2h_pts[h] += 1; h2h_pts[a] += 1
            else:           # Away win
                as_ = random.randint(1, 3)
                hs_ = random.randint(0, max(0, as_-1))
                pts[a] += 3; h2h_pts[a] += 3

            diff = hs_ - as_
            gd[h] += diff;  gd[a] -= diff
            gf[h] += hs_;   gf[a] += as_
            ga[h] += as_;   ga[a] += hs_
            h2h_gd[h] += diff; h2h_gd[a] -= diff

    # Proper FIFA tiebreaker: overall pts → H2H pts → H2H GD → GD → GF → random
    ranked = sorted(teams,
        key=lambda t: (pts[t], h2h_pts[t], h2h_gd[t], gd[t], gf[t], random.random()),
        reverse=True)
    return ranked, pts, gd, gf


def simulate_ko_match(h, a):
    """Knockout match — no draws. Redistribute draw probability (extra time/penalties)."""    hw, d, aw = MATCHUPS.get((h, a), (0.4, 0.2, 0.4))
    p_h = hw + d/2; p_a = aw + d/2
    return h if random.random() < p_h/(p_h+p_a) else a


print("Simulation functions ready.")


In [ ]:
N_SIMS = 10_000
win_counts = defaultdict(int)
sf_counts  = defaultdict(int)
f_counts   = defaultdict(int)

print(f"Running {N_SIMS:,} simulations...")

for sim in range(N_SIMS):
    qualifiers = []; thirds = []

    # Group stage — 12 groups of 4
    for _, teams in WC2026_GROUPS.items():
        ranked, pts, gd, gf = simulate_group(teams)
        qualifiers.append(ranked[0])  # 1st
        qualifiers.append(ranked[1])  # 2nd
        thirds.append((ranked[2], pts[ranked[2]], gd[ranked[2]], gf[ranked[2]]))

    # Best 8 third-placed teams (12 groups × 1 third-placed = 12 teams → top 8)
    thirds.sort(key=lambda x: (-x[1], -x[2], -x[3], random.random()))
    qualifiers += [x[0] for x in thirds[:8]]  # 24 + 8 = 32 teams

    random.shuffle(qualifiers)
    bracket = list(qualifiers)

    # Knockout rounds: R32 → R16 → QF → SF → F
    rnd = 0
    while len(bracket) > 1:
        if rnd == 3:  # Semifinal
            for t in bracket: sf_counts[t] += 1
        if rnd == 4:  # Final
            for t in bracket: f_counts[t] += 1
        bracket = [simulate_ko_match(bracket[i], bracket[i+1])
                   for i in range(0, len(bracket)-1, 2)]
        rnd += 1

    win_counts[bracket[0]] += 1

print("Simulation complete.")


In [ ]:
print(f"\n{'Rank':<5} {'Team':<25} {'Win%':>7} {'Final%':>8} {'Semi%':>8}")
print("-" * 55)
ranked_results = sorted(ALL_TEAMS, key=lambda t: -win_counts[t])
for rank, team in enumerate(ranked_results[:20], 1):
    wp  = win_counts[team] / N_SIMS * 100
    fp  = f_counts[team]   / N_SIMS * 100
    sfp = sf_counts[team]  / N_SIMS * 100
    print(f"{rank:<5} {team:<25} {wp:>6.2f}% {fp:>7.2f}% {sfp:>7.2f}%")

print(f"\nRandom baseline (equal probability): {100/48:.2f}%")


## 8. Visualizations

In [ ]:
# Chart 1: Full Win Probability — all teams by confederation
CONF_COLORS = {
    "UEFA":"#1a78cf","CONMEBOL":"#2ecc71","CAF":"#e67e22",
    "AFC":"#e74c3c","CONCACAF":"#27ae60","OFC":"#95a5a6",
}

all_teams_sorted = sorted(ALL_TEAMS, key=lambda t: -win_counts[t])
probs_sorted = [win_counts[t]/N_SIMS*100 for t in all_teams_sorted]
conf_colors  = [CONF_COLORS.get(CONFEDERATION_MAP.get(t,"?"),"#888") for t in all_teams_sorted]

fig, ax = plt.subplots(figsize=(11, max(8, len(all_teams_sorted)*0.38)))
bars = ax.barh(all_teams_sorted, probs_sorted, color=conf_colors, edgecolor="white", linewidth=0.4)
ax.axvline(100/48, color="gold", linestyle="--", linewidth=1.2,
           alpha=0.8, label=f"Random baseline ({100/48:.1f}%)")
for bar, val in zip(bars, probs_sorted):
    if val >= 0.4:
        ax.text(val+0.1, bar.get_y()+bar.get_height()/2,
                f"{val:.1f}%", va="center", fontsize=8, color="white")
ax.set_xlabel("Win Probability (%)")
ax.set_title(f"FIFA World Cup 2026 — Tournament Win Probabilities\n{N_SIMS:,} Monte Carlo Simulations",
             fontsize=13, fontweight="bold")
patches = [mpatches.Patch(color=v, label=k) for k,v in CONF_COLORS.items()]
ax.legend(handles=patches + [ax.lines[0]], loc="lower right", fontsize=9)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("outputs/win_probabilities_all.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/win_probabilities_all.png")


In [ ]:
# Chart 2: Top 10 detailed — Win / Final / Semi probabilities
top10 = ranked_results[:10]
x = np.arange(len(top10)); width = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width, [win_counts[t]/N_SIMS*100 for t in top10],
       width, label="Win %", color="#c8a84b", edgecolor="white")
ax.bar(x,         [f_counts[t]/N_SIMS*100 for t in top10],
       width, label="Final %", color="#3498db", edgecolor="white")
ax.bar(x + width, [sf_counts[t]/N_SIMS*100 for t in top10],
       width, label="Semi %", color="#2ecc71", edgecolor="white")
ax.set_xticks(x); ax.set_xticklabels(top10, rotation=30, ha="right")
ax.set_ylabel("Probability (%)")
ax.set_title("Top 10 Teams — Win / Final / Semi Probabilities", fontsize=12, fontweight="bold")
ax.legend(); ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("outputs/top10_probabilities.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/top10_probabilities.png")


In [ ]:
# Chart 3: Model comparison — v1 vs v2
model_names = ["LR (v1)", "LR (v2)", "LR Cal.", "XGB Cal.", "LGB Cal.", "RF Cal.", "Ensemble"]
log_losses   = [0.8823,
                model_store["LR (original)"]["ll"],
                model_store["LR (calibrated)"]["ll"],
                model_store["XGBoost (calibrated)"]["ll"],
                model_store["LightGBM (calibrated)"]["ll"],
                model_store["Random Forest (calibrated)"]["ll"],
                ens_ll]
accuracies   = [0.5982,
                model_store["LR (original)"]["acc"],
                model_store["LR (calibrated)"]["acc"],
                model_store["XGBoost (calibrated)"]["acc"],
                model_store["LightGBM (calibrated)"]["acc"],
                model_store["Random Forest (calibrated)"]["acc"],
                ens_acc]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Model Comparison — v1 Baseline vs v2 Upgrades", fontsize=13, fontweight="bold")

colors = ["#e74c3c" if i==0 else "#2ecc71" if i==1 else "#3498db" for i in range(len(model_names))]
bars1 = ax1.bar(model_names, log_losses, color=colors, edgecolor="white")
ax1.set_title("Log-Loss (lower = better)"); ax1.set_ylabel("Log-Loss")
for bar, val in zip(bars1, log_losses):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
             f"{val:.4f}", ha="center", fontsize=9)
ax1.tick_params(axis="x", rotation=30)
ax1.spines[["top","right"]].set_visible(False)

bars2 = ax2.bar(model_names, [a*100 for a in accuracies], color=colors, edgecolor="white")
ax2.set_title("Accuracy (%) (higher = better)"); ax2.set_ylabel("Accuracy (%)")
for bar, val in zip(bars2, accuracies):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
             f"{val*100:.1f}%", ha="center", fontsize=9)
ax2.tick_params(axis="x", rotation=30)
ax2.spines[["top","right"]].set_visible(False)

from matplotlib.patches import Patch
legend_elements = [Patch(color="#e74c3c",label="v1 baseline"),
                   Patch(color="#2ecc71",label="v2 best"),
                   Patch(color="#3498db",label="v2 other")]
ax1.legend(handles=legend_elements, fontsize=9)
plt.tight_layout()
plt.savefig("outputs/model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/model_comparison.png")
